# SPA: Challenger

Este notebook comprueba si un modelo ajustado supera al champion, la regresión logística con valores por defecto. Primero reproduce el split del champion y lo verifica, porque el champion evaluado sobre este test da la misma PR-AUC que se registró. Después exporta gold_split.csv, que marca cada cliente como train o test, para la búsqueda de hiperparámetros en local (11a_hyperparameter_search). Random forest, gradient boosting y XGBoost se entrenan con los hiperparámetros elegidos allí, con los mismos folds que el champion, y se registran en MLflow. Los tres superan a la regresión logística por ~0.02 de PR-AUC, pero la ventaja no es consistente: XGBoost, el mejor, gana en 4 de 5 folds y pierde en uno. Además, los tres modelos de árboles ganan y pierden en los mismos folds, así que la diferencia depende más de qué clientes caen en cada fold que del algoritmo. Con la misma lista de llamadas, XGBoost encuentra 982 de los 1495 clientes que se van y la regresión logística 964, unos 12 más por cada 1000.

Decisión: se mantiene la regresión logística como champion. La mejora no compensa perder la explicabilidad de sus coeficientes ni volver a desplegar. Los modelos ajustados quedan registrados como runs en MLflow, no en Unity Catalog, y el test no se usa porque no se promueve ningún modelo.

| Fold | Random forest | Gradient boosting | XGBoost |
|---|---|---|---|
| 1 | +0.041 | +0.039 | +0.047 |
| 2 | +0.042 | +0.035 | +0.040 |
| 3 | −0.006 | +0.012 | +0.010 |
| 4 | −0.010 | −0.008 | −0.003 |
| 5 | +0.031 | +0.014 | +0.023 |
| **Media/mean** | **+0.020** | **+0.018** | **+0.023** |

# ENG: Challenger

This notebook checks whether a tuned model beats the champion, the logistic regression with default values. It first reproduces the champion's split and verifies it, since the champion evaluated on this test set gives the same PR-AUC that was logged. It then exports gold_split.csv, which marks each customer as train or test, for the local hyperparameter search (11a_hyperparameter_search). Random forest, gradient boosting and XGBoost are trained with the hyperparameters chosen there, using the same folds as the champion, and logged in MLflow. All three beat logistic regression by ~0.02 PR-AUC, but the advantage is not consistent: XGBoost, the best, wins in 4 of 5 folds and loses in one. In addition, the three tree models win and lose in the same folds, so the difference depends more on which customers fall into each fold than on the algorithm. With the same call list, XGBoost finds 982 of the 1495 customers who leave and logistic regression 964, about 12 more per 1000.

Decision: logistic regression remains the champion. The improvement does not justify losing the explainability of its coefficients or redeploying. The tuned models stay logged as runs in MLflow, not in Unity Catalog, and the test set is not used because no model is promoted.

In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [0]:
import numpy as np
import mlflow
from mlflow import MlflowClient
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score
from mlflow.models import infer_signature
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from churn.pipeline import construir_pipeline
from churn.config import MLFLOW_EXPERIMENT_NAME

from churn.config import GOLD_TABLE, MODEL_NAME, ID, TARGET, RANDOM_SEED, EXPORTS_VOLUME
from churn.validation import validar_gold

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

In [0]:
usuario = spark.sql("SELECT current_user()").collect()[0][0]
experiment = mlflow.set_experiment(f"/Users/{usuario}/{MLFLOW_EXPERIMENT_NAME}")

In [0]:
import sys
sys.version

In [0]:
pdf = spark.table(GOLD_TABLE).toPandas()
validar_gold(pdf)

X = pdf.drop(columns=[ID, TARGET])
y = pdf[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

assert np.abs(y_train.mean() - y_test.mean()) < 0.01, "La proporción de churn en train y test no es la misma"

In [0]:
champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
modelo = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/{champion.version}")

pr_auc_ahora = average_precision_score(y_test, modelo.predict(X_test)[:, 1])
pr_auc_registrada = client.get_run(champion.run_id).data.metrics["pr_auc_test"]

assert np.isclose(pr_auc_ahora, pr_auc_registrada), f"Split distinto: {pr_auc_ahora} frente a {pr_auc_registrada}"

In [0]:
gold_split = pdf.copy()
gold_split["conjunto"] = "train"
gold_split.loc[X_test.index, "conjunto"] = "test"

gold_split.to_csv(f"{EXPORTS_VOLUME}/gold_split.csv", index=False)

In [0]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

candidatos = {
    "rf_ajustado": RandomForestClassifier(
        n_estimators=600, max_depth=8, min_samples_leaf=10, max_features=0.3,
        random_state=RANDOM_SEED),
    "gb_ajustado": GradientBoostingClassifier(
        n_estimators=400, learning_rate=0.03, max_depth=2, min_samples_leaf=200, subsample=1.0,
        random_state=RANDOM_SEED),
    "xgb_ajustado": XGBClassifier(
        n_estimators=600, learning_rate=0.01, max_depth=2, min_child_weight=5,
        subsample=0.7, colsample_bytree=0.7, reg_lambda=1,
        random_state=RANDOM_SEED),
}

In [0]:
def entrenar_y_registrar(modelo, nombre_run, params):
    pipe = construir_pipeline(modelo)

    with mlflow.start_run(run_name=nombre_run) as run:
        mlflow.log_params(params)

        scores = cross_val_score(pipe, X_train, y_train, cv=cv,
                                 scoring="average_precision")
        mlflow.log_metric("pr_auc_cv_media", scores.mean())
        mlflow.log_metric("pr_auc_cv_std", scores.std())

        pipe.fit(X_train, y_train)
        ejemplo = X_train.head(5)
        firma = infer_signature(ejemplo, pipe.predict_proba(ejemplo))
        info = mlflow.sklearn.log_model(
            sk_model=pipe, name="modelo", signature=firma,
            input_example=ejemplo, pyfunc_predict_fn="predict_proba",
        )

    return run.info.run_id, info.model_uri, scores

In [0]:
resultados = {}
for nombre, modelo in candidatos.items():
    params = {"modelo": type(modelo).__name__, "config": "ajustado", **modelo.get_params()}
    resultados[nombre] = entrenar_y_registrar(modelo, nombre, params)

In [0]:
for nombre, (run_id, uri, scores) in resultados.items():
    print(nombre, scores.mean(), scores.std())

In [0]:
logreg = construir_pipeline(LogisticRegression(random_state=RANDOM_SEED))
scores_logreg = cross_val_score(logreg, X_train, y_train, cv=cv, scoring="average_precision")

run_logreg = mlflow.search_runs(filter_string="tags.mlflow.runName = 'logreg_base'")
media_registrada = run_logreg["metrics.pr_auc_cv_media"].iloc[0]

assert np.isclose(scores_logreg.mean(), media_registrada), f"{scores_logreg.mean()} frente a {media_registrada}"

In [0]:
import pandas as pd

folds = pd.DataFrame({nombre: scores for nombre, (_, _, scores) in resultados.items()})
diferencias = folds.sub(scores_logreg, axis=0)
diferencias

In [0]:
from sklearn.model_selection import cross_val_predict
from churn.config import UMBRAL

xgb_pipe = construir_pipeline(candidatos["xgb_ajustado"])

oof_logreg = cross_val_predict(logreg, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
oof_xgb = cross_val_predict(xgb_pipe, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

In [0]:
tabla = pd.DataFrame({"churn": y_train.values, "logreg": oof_logreg, "xgb": oof_xgb})

n_llamadas = int((tabla["logreg"] >= UMBRAL).sum())

for modelo in ["logreg", "xgb"]:
    print(modelo, tabla.nlargest(n_llamadas, modelo)["churn"].sum())

print("Total que se van:", tabla["churn"].sum())
print("Total de llamadas:", n_llamadas)